In [33]:
import torch
import math
import torch.nn as nn

# This cell imports the necessary libraries for building the transformer model:
#   - torch: The main PyTorch library for tensor operations and neural networks.
#   - math: Used for mathematical functions like `sqrt` in the attention mechanism.
#   - torch.nn: PyTorch's neural network module, although custom layers are implemented here.

In [34]:
# This cell checks for the availability of a GPU and sets the device accordingly.
# Using a GPU can significantly speed up training for deep learning models.

# Check if CUDA (NVIDIA GPU support) is available, otherwise use the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print the selected device.
print("Device:", device)

# If a GPU is being used, print additional information about it.
if device.type == "cuda":
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
    print("Total GPU Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

Device: cpu


In [35]:
# This class implements a basic Linear (fully connected) layer.
# It performs a linear transformation: y = x @ W + b.
class Linear:
    def __init__(self, in_features, out_features):
        # Initialize weights (W) with small random values and biases (b) with zeros.
        # Weights are initialized with a small scale (0.02) to prevent large initial gradients.
        self.W = torch.randn(in_features, out_features, device=device) * 0.02
        self.b = torch.zeros(out_features, device=device)

        # Enable gradient computation for weights and biases, as they are learnable parameters.
        self.W.requires_grad = True
        self.b.requires_grad = True

    def __call__(self, x):
        # Perform the linear transformation.
        return x @ self.W + self.b

    def parameters(self):
        # Return a list of learnable parameters for this layer.
        return [self.W, self.b]

In [36]:

# This class implements an Embedding layer, which converts integer indices into dense vectors.
# It's commonly used to represent words or tokens in a fixed-size vector space.
class Embedding:
    def __init__(self, vocab_size, d_model):
        # Initialize the embedding matrix (weight) with small random values.
        # vocab_size: the number of unique tokens in the vocabulary.
        # d_model: the dimensionality of the embedding vectors.
        self.weight = torch.randn(vocab_size, d_model, device=device) * 0.02
        # Enable gradient computation for the embedding matrix.
        self.weight.requires_grad = True

    def __call__(self, x):
        # Look up the embedding vectors for the given input indices 'x'.
        return self.weight[x]

    def parameters(self):
        # Return the learnable embedding matrix.
        return [self.weight]

In [37]:

# This class implements Positional Encoding, which adds information about the position
# of tokens in a sequence. Transformers are permutation-invariant, so positional
# encoding is crucial for maintaining sequential order.
class PositionalEncoding:
    def __init__(self, d_model, max_len=5000):
        # Initialize an encoding matrix with zeros.
        # d_model: the dimensionality of the model (same as embedding dimension).
        # max_len: the maximum sequence length the model is expected to handle.
        self.encoding = torch.zeros(max_len, d_model, device=device)
        # Positional encodings are typically not learnable, so gradients are disabled.
        self.encoding.requires_grad = False

        # Create a tensor for positions (0, 1, ..., max_len-1).
        pos = torch.arange(0, max_len, device=device).float().unsqueeze(1)
        # Create a tensor for the '2i' part of the formula for sin/cos arguments.
        _2i = torch.arange(0, d_model, step=2, device=device).float()

        # Apply the sine function to even indices.
        self.encoding[:, 0::2] = torch.sin(pos / (10000 ** (_2i / d_model)))
        # Apply the cosine function to odd indices.
        self.encoding[:, 1::2] = torch.cos(pos / (10000 ** (_2i / d_model)))

    def __call__(self, x):
        # Add the positional encoding to the input tensor 'x'.
        # It takes a slice of the encoding matrix corresponding to the input sequence length.
        return x + self.encoding[:x.size(1)]

In [38]:

# This function implements the core scaled dot-product attention mechanism.
# It calculates attention weights based on similarity between Query (Q) and Key (K),
# and then applies these weights to Value (V).
def attention(Q, K, V, mask=None):
    # Get the dimension of the keys (d_k).
    d_k = Q.size(-1)

    # Calculate raw attention scores: Q @ K_transpose / sqrt(d_k).
    # Scaling by sqrt(d_k) prevents the dot products from growing too large,
    # which can push the softmax into regions with very small gradients.
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)

    # If a mask is provided (e.g., for padding or preventing future information leakage),
    # fill masked positions with a very large negative number (-1e9).
    # This makes their softmax probability effectively zero.
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    # Apply softmax to get attention weights. The sum of weights for each query will be 1.
    weights = torch.softmax(scores, dim=-1)
    # Multiply weights by Value (V) to get the final attention output.
    return weights @ V

In [39]:

# This class implements Multi-Head Attention, which allows the model to jointly attend
# to information from different representation subspaces at different positions.
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        # Ensure that d_model is divisible by num_heads, as each head gets d_model / num_heads dimensions.
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        # d_k (or d_v) is the dimension of the keys, queries, and values for each head.
        self.d_k = d_model // num_heads

        # Linear layers for transforming input into Queries, Keys, and Values for all heads.
        self.W_q = Linear(d_model, d_model)
        self.W_k = Linear(d_model, d_model)
        self.W_v = Linear(d_model, d_model)
        # Output linear layer to combine the outputs from all attention heads.
        self.W_o = Linear(d_model, d_model)

    def split_heads(self, x):
        # Reshapes the input tensor to prepare it for multi-head attention.
        # B: batch size, T: sequence length, D: d_model.
        B, T, D = x.shape
        # Reshape to (B, T, num_heads, d_k) and then transpose to (B, num_heads, T, d_k).
        # This puts the heads dimension second, which is convenient for parallel attention computation.
        x = x.view(B, T, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def combine_heads(self, x):
        # Combines the outputs from multiple attention heads back into a single tensor.
        # B: batch size, H: num_heads, T: sequence length, D: d_k.
        B, H, T, D = x.shape
        # Transpose back to (B, T, num_heads, d_k) and then use .contiguous() to ensure memory layout
        # is correct before reshaping.
        x = x.transpose(1, 2).contiguous()
        # Reshape to (B, T, d_model) by concatenating the outputs of all heads.
        return x.view(B, T, H * D)

    def __call__(self, x):
        # Apply linear transformations and split into multiple heads for Q, K, V.
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # Perform scaled dot-product attention for all heads in parallel.
        attn = attention(Q, K, V)
        # Combine the outputs from all heads.
        out = self.combine_heads(attn)

        # Apply the final linear layer to the combined output.
        return self.W_o(out)

    def parameters(self):
        # Return a list of all learnable parameters from the linear layers within MultiHeadAttention.
        return (
            self.W_q.parameters() +
            self.W_k.parameters() +
            self.W_v.parameters() +
            self.W_o.parameters()
        )

In [40]:

# This class implements a simple Feed-Forward Network (FFN) with a ReLU activation.
# It consists of two linear transformations with a non-linearity in between.
class FeedForward:
    def __init__(self, d_model, d_ff):
        # fc1: First linear layer, maps from d_model to d_ff (inner dimension).
        self.fc1 = Linear(d_model, d_ff)
        # fc2: Second linear layer, maps from d_ff back to d_model.
        self.fc2 = Linear(d_ff, d_model)

    def __call__(self, x):
        # Apply the first linear layer, then ReLU activation, then the second linear layer.
        return self.fc2(torch.relu(self.fc1(x)))

    def parameters(self):
        # Return all learnable parameters from both linear layers.
        return self.fc1.parameters() + self.fc2.parameters()

In [41]:

# This class implements Layer Normalization, which normalizes the inputs across the
# feature dimension, helping to stabilize and speed up training.
class LayerNorm:
    def __init__(self, d_model, eps=1e-5):
        # gamma and beta are learnable parameters that scale and shift the normalized output.
        # They are initialized to 1s and 0s respectively.
        self.gamma = torch.ones(d_model, device=device, requires_grad=True)
        self.beta = torch.zeros(d_model, device=device, requires_grad=True)
        # eps is a small value added to the variance to prevent division by zero.
        self.eps = eps

    def __call__(self, x):
        # Calculate the mean and variance along the last dimension (feature dimension).
        # keepdim=True maintains the dimension for broadcasting during normalization.
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        # Normalize the input: (x - mean) / sqrt(var + eps).
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        # Apply the learnable scaling (gamma) and shifting (beta).
        return self.gamma * x_norm + self.beta

    def parameters(self):
        # Return the learnable gamma and beta parameters.
        return [self.gamma, self.beta]

In [42]:

# This class implements a single Encoder Layer, which is a fundamental building block
# of the Transformer encoder. It consists of Multi-Head Attention and a Feed-Forward Network,
# each followed by a residual connection and Layer Normalization.
class EncoderLayer:
    def __init__(self, d_model, num_heads, d_ff):
        # Multi-Head Attention mechanism.
        self.attn = MultiHeadAttention(d_model, num_heads)
        # Layer Normalization after the first sub-layer (attention + residual).
        self.norm1 = LayerNorm(d_model)

        # Feed-Forward Network.
        self.ff = FeedForward(d_model, d_ff)
        # Layer Normalization after the second sub-layer (FFN + residual).
        self.norm2 = LayerNorm(d_model)

    def __call__(self, x):
        # First sub-layer: Multi-Head Attention + Residual Connection + Layer Normalization.
        # The input 'x' is added to the attention output (residual connection),
        # and then normalized.
        x = self.norm1(x + self.attn(x))
        # Second sub-layer: Feed-Forward Network + Residual Connection + Layer Normalization.
        # The output from the first sub-layer is added to the FFN output (residual connection),
        # and then normalized.
        x = self.norm2(x + self.ff(x))
        return x

    def parameters(self):
        # Return all learnable parameters from the attention, FFN, and normalization layers.
        return (
            self.attn.parameters() +
            self.norm1.parameters() +
            self.ff.parameters() +
            self.norm2.parameters()
        )

In [43]:

# This class implements the full Transformer Encoder model.
# It combines embedding, positional encoding, multiple encoder layers, and a final linear output layer.
class Transformer:
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff):
        # Embedding layer to convert input token IDs into dense vectors.
        self.embed = Embedding(vocab_size, d_model)
        # Positional Encoding to inject sequence order information.
        self.pos = PositionalEncoding(d_model)

        # Stack of 'num_layers' EncoderLayer instances.
        self.layers = [
            EncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ]

        # Final linear layer to project the transformer output back to the vocabulary size
        # for tasks like next-token prediction or classification.
        self.fc_out = Linear(d_model, vocab_size)

    def __call__(self, x):
        # Apply embedding and positional encoding to the input.
        x = self.embed(x)
        x = self.pos(x)

        # Pass the input through each encoder layer sequentially.
        for layer in self.layers:
            x = layer(x)

        # Apply the final linear output layer.
        return self.fc_out(x)

    def parameters(self):
        # Collect all learnable parameters from the embedding, all encoder layers, and the final output layer.
        params = []
        params += self.embed.parameters()

        for layer in self.layers:
            params += layer.parameters()

        params += self.fc_out.parameters()
        return params

In [44]:

# This function implements the cross-entropy loss, commonly used for classification tasks.
# It measures the difference between the predicted logits and the true target labels.
def cross_entropy(logits, targets):
    # Reshape logits to (batch_size * sequence_length, vocab_size)
    # and targets to (batch_size * sequence_length) for easier calculation.
    logits = logits.view(-1, logits.size(-1))
    targets = targets.view(-1)

    # Calculate log probabilities using log_softmax.
    log_probs = torch.log_softmax(logits, dim=-1)
    # Select the log probability corresponding to the true target for each position
    # and then take the negative mean across all positions to get the loss.
    return -log_probs[range(len(targets)), targets].mean()

In [45]:
# This cell defines a simple training loop for the Transformer model.

# Define vocabulary size.
vocab_size = 1000

# Instantiate the Transformer model with specified parameters.
model = Transformer(
    vocab_size=vocab_size,
    d_model=64,       # Dimensionality of the model's representations.
    num_heads=4,      # Number of attention heads in MultiHeadAttention.
    num_layers=2,     # Number of EncoderLayer blocks.
    d_ff=128          # Dimensionality of the inner layer of the FeedForward Network.
)

# Set the learning rate for the optimizer (manual gradient update in this case).
lr = 1e-3

# Training loop for a fixed number of steps.
for step in range(50):
    # Generate random input (x) and target (y) sequences.
    # (2, 10) means a batch size of 2 and sequence length of 10.
    # The values are random integers within the vocabulary size.
    x = torch.randint(0, vocab_size, (2, 10), device=device)
    y = torch.randint(0, vocab_size, (2, 10), device=device)

    # Forward pass: get model predictions (logits).
    logits = model(x)
    # Calculate the cross-entropy loss between predictions and targets.
    loss = cross_entropy(logits, y)

    # Backward pass: compute gradients for all learnable parameters.
    loss.backward()

    # Manually update parameters using gradient descent.
    # This iterates through all parameters of the model.
    for p in model.parameters():
        # Update parameter value: p_new = p_old - learning_rate * p_grad.
        p.data -= lr * p.grad
        # Zero out gradients after updating to prevent accumulation for the next step.
        p.grad.zero_()

    # Print the loss periodically (every 10 steps).
    if step % 10 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

Step 0, Loss: 6.933408260345459
Step 10, Loss: 6.905920505523682
Step 20, Loss: 6.9418535232543945
Step 30, Loss: 6.9561567306518555
Step 40, Loss: 6.932481288909912
